# 01 — Data Understanding: the Royal Court's ledger

**Purpose.** Before a fraud-scoring model can judge a new loan, the Royal Court must know what is written in its ledger, which records have an observed outcome, and which fields would reveal the future. This notebook establishes that foundation. It does not train a model.

The local source used below is data/raw/sample.xlsx. The roadmap uses FPD_15 on applications with cenz_15_1_maturity_flag = 1 as the primary **delinquency-based fraud/risk proxy**. A positive FPD_15 is not proof of fraud.

**How this helps the pipeline.** Your findings become the field and leakage rules for EDA, data quality, feature engineering, and later temporal validation. Keep observations separate from assumptions; the crystal ball cannot be trusted if its scrolls are misread.

## Learning objectives and assignment map

By the end of this notebook, you should be able to:

- Describe the source, row count, columns, data types, and meaning of the main fields.
- Distinguish application IDs, candidate predictors, cohort filters, outcomes, and probable leakage.
- Check record identity, duplicates, and missingness without losing the original context.
- Explain how many eligible applications and FPD positives are available, with a clear denominator.
- Summarize the products and cities represented and write cautious, evidence-based observations.

Work through six tasks below: **source overview → field translation → schema and keys → missingness → product/city inventory → target and leakage audit**. Existing code and outputs are your starting point. A task is complete when its interpretation and acceptance criteria are met, not when code has merely run.

**Notebook-wide deliverables.** A compact dataset summary, a field-role table, key and missingness findings, a mature-target summary, and a brief final handoff to notebook 02.

### How to use this assignment

Preserve the original ledger when checking identity: compare application IDs and full rows before interpreting duplicate counts after any column is removed. For every reported percentage, name its denominator. If a field's meaning is uncertain, record the uncertainty rather than turning it into a feature assumption.

The questions and checklists guide your own work. Add your own analysis or explanation where a criterion is not yet met; no model or complete feature pipeline belongs in this notebook.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl

## Task 1 — Open and describe the ledger

**Learning objective.** Identify the data source and establish the unit and size of the analysis.

**Purpose and relevance.** Every later rate and split depends on knowing what one row represents and which workbook was read.

**Tasks.**
- [ ] Confirm the workbook and sheet being loaded and inspect a few rows and the column names.
- [ ] Report the number of application rows and source columns.
- [ ] State what an application row represents and the date range covered.

**Expected deliverable.** A short source summary with dimensions and one observation about the raw fields.

**Acceptance criteria.** The current local source is identified as data/raw/sample.xlsx; the observed shape is 25,536 rows by 20 columns, or any difference is explained. The summary does not call all rows eligible FPD observations.

**Think about it.** Why can a workbook contain more applications than the population used to evaluate FPD_15?

**Before moving on:** [ ] source identified  [ ] unit and shape stated  [ ] date range stated

In [2]:
df = pd.read_excel('../data/raw/sample.xlsx')
df.head()

,Дата заявки,Код заявки,Дата УДЛ клиента,Кредитный продукт,Заявленный официальный доход с анкеты,Менеджер,Запрошенная сумма кредита,Город,БИН работодаетя с отчета с ГЦВП,Просрочки за последние 6 месяцев в днях с кредитной истории ПКБ,Кол-во завершенных займов с кредитной истории ПКБ,Сумма остатка по всем займам с кредитной истории ПКБ,Кол-ао действующих займов клиента кредитной истории ПКБ,Кол-во запросов в ПКБ за последние 30 дней,Срок займа в мес,Признак вызревания CENZ_15_1,FPD_15,Признак вызревания CENZ_15_2,SPD_15,Активная просрочка в днях
0,2022-06-24,KAR.00095934,2006-12-14,Автокредитов,308260.0,Налыбек,2800000,Караганд,NaN,0.0,2.0,2290527.0,3,77.0,NaN,1,0,1,0,0
1,2022-06-23,1003534418,2013-01-10,Кредит налич,270000.0,Уштенова,1500000,Нур-Султ,990240003263/,0.0,9.0,2016655.0,4,36.0,5.0,1,0,1,0,0
2,2022-05-26,BCC_5407,2017-09-20,#картакарта_v,180000.0,Пользователь,70000,Тараз,991040000928/,9.0,4.0,347444.0,2,27.0,11.0,0,0,0,0,0
3,2022-05-15,BCC_1976,2006-08-28,#картакарта_v,800000.0,Пользователь,200000,Алматы,981141001567/,0.0,9.0,6297948.0,10,26.0,6.0,0,0,0,0,0
4,2022-04-29,ALM.00285810,2021-02-19,Автокредитов,600000.0,Мешітбай,11500000,Алматы,890111301342/,2.0,7.0,1974935.0,4,25.0,3.0,1,0,1,0,39


In [3]:
df.shape

(25536, 20)

## Task 2 — Translate the column scrolls

**Learning objective.** Map the original labels to readable names while retaining their meaning.

**Purpose and relevance.** Clear names make the next audits easier, but a misleading translation can create a bad feature or an incorrect target definition.

**Tasks.**
- [ ] Review the mapping for all 20 source fields and confirm no field was lost or accidentally given a duplicate name.
- [ ] Identify dates, IDs, categorical fields, numeric fields, cohort flags, and outcomes.
- [ ] Mark any ambiguous field meaning for verification; in particular, do not assume the ID-document date is a birth date.

**Expected deliverable.** A source-to-renamed-field map or concise field dictionary with a provisional role for each field.

**Acceptance criteria.** All 20 fields are accounted for, the renamed schema is inspectable, and unclear meanings are documented rather than guessed.

**Think about it.** If a scroll says when a document expires, can it tell you the Knight's age?

**Before moving on:** [ ] names checked  [ ] field roles drafted  [ ] ambiguous meanings flagged

In [4]:
rename_map = {
    'Дата заявки': 'application_date',
    'Код заявки': 'application_id',
    'Дата УДЛ клиента': 'client_id_document_date',
    'Кредитный продукт': 'credit_product',
    'Заявленный официальный доход с анкеты': 'declared_official_income',
    'Менеджер': 'manager',
    'Запрошенная сумма кредита': 'requested_loan_amount',
    'Город': 'city',
    'БИН работодаетя с отчета с ГЦВП': 'employer_bin_gcvp',
    'Просрочки за последние 6 месяцев в днях с кредитной истории ПКБ': 'delinquency_days_6m_credit_bureau',
    'Кол-во завершенных займов с кредитной истории ПКБ': 'completed_loans_count_credit_bureau',
    'Сумма остатка по всем займам с кредитной истории ПКБ': 'total_outstanding_balance_credit_bureau',
    'Кол-ао действующих займов клиента кредитной истории ПКБ': 'active_loans_count_credit_bureau',
    'Кол-во запросов в ПКБ за последние 30 дней': 'credit_bureau_inquiries_30d',
    'Срок займа в мес': 'loan_term_months',
    'Признак вызревания CENZ_15_1': 'cenz_15_1_maturity_flag',
    'FPD_15': 'fpd_15',
    'Признак вызревания CENZ_15_2': 'cenz_15_2_maturity_flag',
    'SPD_15': 'spd_15',
    'Активная просрочка в днях': 'active_delinquency_days',
}

df = df.rename(columns=rename_map)

## Field guide — The Royal Court's twenty scrolls

Welcome back, Treasurer. The DataFrame is the Ancient Scroll of the Royal Court. Each Knight's application carries clues, but a clue is not a conviction. The Master of Whispers must separate information available when the Knight arrives from the **prophecies of the future** that would let the crystal ball cheat.

### 🛡️ The Identity Scrolls

- **application_id — the Quest Scroll Number:** use it to trace and check applications. A repeated number deserves investigation; the number itself is not a model predictor.
- **application_date — the Day the Knight Arrived:** known at application time and useful later for calendar features and chronological validation.
- **client_id_document_date — the Knight's document scroll:** the English workbook calls this Client_ID_Expiry_Date. It is **not a birth scroll**. Verify its meaning before deriving anything from it. The current sample has 261 missing values, not 25,275.

### 🏰 The Knight's Manifest

- **credit_product — the Type of Quest:** imaginary “Dragon Slaying” and “Castle Repair” loans illustrate how products may differ; use the real categories in the ledger.
- **manager — the Gatekeeper:** inspect the field, but the roadmap reserves it for a separate experiment. A pattern by manager would not prove misconduct.
- **city — the Kingdom of Origin:** count the represented places before comparing outcomes; a small kingdom can have a noisy rate.
- **employer_bin_gcvp — the Knight's Lord:** an employer identifier with many categories. The sample has 3,747 missing values, not 21,789; missingness may be informative but is not proof of risk.
- **declared_official_income — claimed Gold Earnings:** a candidate numeric field. A Knight claiming 100 gold while requesting 10,000 gold raises a question, not a verdict.
- **requested_loan_amount — the Gold They Desire:** a candidate numeric field; compare its distribution and scale with income in later notebooks.
- **loan_term_months — the Length of the Quest:** a candidate field with missing values whose meaning and quality must be checked.

### 🔮 The Credit Bureau's spy network

These records describe prior borrowing, provided they were available when the application was made. Do not assume a relationship with FPD until you measure it.

- **delinquency_days_6m_credit_bureau — Days of Shame in the last six moons:** recent prior overdue days.
- **completed_loans_count_credit_bureau — completed Quests:** number of completed loans.
- **total_outstanding_balance_credit_bureau — current Gold owed:** balance across loans in the bureau history.
- **active_loans_count_credit_bureau — active Quests:** current count of active loans. One nonnumeric blank makes the loaded column an object, so inspect it before classifying it as numeric.
- **credit_bureau_inquiries_30d — Castles Visited Recently:** number of recent bureau inquiries; “desperation meter” is only a metaphor, not a finding.

### ⚔️ The Outcome Curses and fate flags

These scrolls reveal or depend on what happened **after** the application. They are not ordinary predictors.

- **fpd_15 — the first-payment curse:** the primary delinquency outcome and a fraud/risk proxy, not evidence that the Knight stole the gold.
- **spd_15 — the second-payment curse:** a separate, optional outcome, not evidence of a “serial thief.”
- **active_delinquency_days — currently hiding from the Guards:** active overdue days. This is probable application-time leakage and must stay out of model inputs.
- **cenz_15_1_maturity_flag and cenz_15_2_maturity_flag — the fate-readiness flags:** use the appropriate flag to select applications whose FPD or SPD outcome has matured. A value of 1 marks the verified mature population in the roadmap; neither flag is a predictor.

### 🧙‍♂️ The Grand Summary

The future crystal ball will use defensible application-time information to estimate FPD_15 on the eligible population. Keep the Quest Scroll Number for tracing, the fate flags for filtering, and the curses out of predictor inputs. Feature creation and model training come later; this notebook's spell is **understanding the ledger**.

---



## Task 3 — Check schema, types, IDs, and duplicates

**Learning objective.** Distinguish a field's stored type from its real meaning and distinguish repeated keys from identical records.

**Purpose and relevance.** A wrong type can hide a numeric field; a duplicate count after dropping the ID does not establish duplicate applications.

**Tasks.**
- [ ] Inspect the renamed columns and data types; classify all 20 source fields by role and expected type.
- [ ] Check missing and repeated application IDs and exact full-row duplicates before dropping the ID.
- [ ] Explain what the duplicate count means after excluding the ID, and investigate the repeated key.

**Expected deliverables.** A field-role/type table and a short record-integrity note.

**Acceptance criteria.** The type table accounts for the mixed active-loan count. For the current sample, reconcile 25,535 unique IDs, one repeated ID, zero exact full-row duplicates, and 15 repeats after the ID is excluded, or explain a different result. Do not call those 15 rows duplicate applications without investigation.

**Think about it.** How could two different Quest Scroll Numbers describe otherwise identical records?

**Before moving on:** [ ] roles and types classified  [ ] ID checked  [ ] duplicate definitions distinguished



## Task 4 — Map missing values

**Learning objective.** Calculate missing counts and rates and explain where information is absent.

**Purpose and relevance.** Missingness shapes the later quality and preprocessing plan. A blank field is not automatically a zero or a sign of fraud.

**Tasks.**
- [ ] Report missing count and percentage for each field, clearly stating whether the ID is still present.
- [ ] Identify the largest gaps and note any fields that appear to go missing together.
- [ ] Write at least two observations tied to the displayed counts.

**Expected deliverable.** A missingness table and short interpretation.

**Acceptance criteria.** The current sample's 261 missing ID-document dates and 3,747 missing employer BINs are reported correctly, or differences are explained. Percentages name their denominator; nonmissing counts are not called missing counts.

**Think about it.** What would you need to check before interpreting missing employer information as a risk signal?

**Before moving on:** [ ] counts and rates shown  [ ] denominator named  [ ] observations written



## Task 5 — Inventory products and cities

**Learning objective.** Describe who and what the dataset covers before deeper EDA.

**Purpose and relevance.** Product and city mix can affect later comparisons, especially when some groups contain few applications or positives.

**Tasks.**
- [ ] Report product counts and the leading city counts; note rare categories.
- [ ] Explain what the city-by-product table shows and what it does not show.
- [ ] State one evidence-based observation about the dataset mix.

**Expected deliverables.** Product and city frequency tables plus a short observation.

**Acceptance criteria.** Counts are tied to the population used, rare groups are identified, and an unfiltered product-by-FPD table is not presented as a mature-cohort risk comparison. Save detailed subgroup-rate analysis for notebook 02.

**Think about it.** Does a large product count imply a high FPD rate?

**Before moving on:** [ ] products counted  [ ] cities counted  [ ] scope of any FPD table clear



## Task 6 — Define the target and close the leakage audit

**Learning objective.** Select an observable label population and explain which fields a future model may see.

**Purpose and relevance.** Counting unmatured applications as negatives or feeding future information to the model would make its evaluation misleading.

**Tasks.**
- [ ] Report total FPD_15 positives and then the mature FPD cohort's positive and negative counts and positive rate.
- [ ] Explain why cenz_15_1_maturity_flag is a filter and why FPD_15 is a proxy rather than confirmed fraud.
- [ ] Record the role of every source field: candidate, ID, filter, target, probable leakage, or separate experiment.
- [ ] Explain why active_delinquency_days is excluded and why manager needs separate scrutiny.
- [ ] End with three to five concise findings and unanswered questions for notebook 02.

**Expected deliverables.** Mature-target summary, field-role/leakage table, and final handoff paragraph.

**Acceptance criteria.** The current source yields 24,142 mature applications, 111 FPD positives, 24,031 negatives, and about 0.46% positives, or discrepancies are explained. Unmatured rows are not treated as known negatives. No target, maturity flag, application ID, or active overdue field is listed as an ordinary predictor.

**Think about it.** Why would the crystal ball look brilliant if it could read the “currently hiding from the Guards” scroll?

**Before moving on:** [ ] mature denominator correct  [ ] proxy limitation stated  [ ] leakage roles complete  [ ] findings written

## Completion checklist and Definition of Done

- [ ] Source, size, and field meanings documented
- [ ] Schema, IDs, duplicates, and missingness interpreted
- [ ] Product and city coverage summarized
- [ ] Mature FPD outcome quantified with the correct denominator
- [ ] All source fields assigned a defensible role
- [ ] Final observations and open questions written in your own words

**Definition of Done.** All six tasks meet their acceptance criteria; the notebook runs from the stated source; every count or rate has a clear population; the medieval analogies are separated from factual conclusions; and the final handoff is ready to guide the EDA notebook. No model training is required here.
